# Smoke-test 5% — extract → full train (1 file, CHẠY 2 LẦN)

**Cách dùng:** đổi `MODE` ở Cell tiếp theo và ĐỔI ACCELERATOR giữa 2 lần chạy.

| Lần | `MODE` | Accelerator | Add Input | Internet |
|---|---|---|---|---|
| 1 – extract | `"extract"` | **None (CPU)** | dataset **ẢNH** (`rwth-phoenix-2014-t`) | ON |
| 2 – train | `"train"` | **GPU T4** | dataset **POSE** bạn tạo từ `poses_5pct.zip` | ON |

**Quy trình:** chạy lần 1 (CPU) → tab **Output** tải `poses_5pct.zip` → **New Dataset** (upload zip, Kaggle tự giải nén) → đổi `MODE="train"`, đổi Accelerator sang GPU, **Add** dataset pose đó → chạy lần 2.

> 5% chỉ để **kiểm tra pipeline chạy được**, KHÔNG dùng số liệu cho báo cáo (BLEU gần sàn, RL là nhiễu).

In [ ]:
# ================== CÔNG TẮC ==================
MODE = "extract"   # <-- lần 1: "extract"  |  lần 2: đổi thành "train"
# ============================================

import sys, subprocess
_pkgs = ["sacrebleu", "sentencepiece", "tqdm"]
if MODE == "extract":
    _pkgs = ["mediapipe==0.10.14", "opencv-python-headless"] + _pkgs
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_pkgs], check=True)
print("MODE =", MODE, "| cài deps xong")

In [ ]:
# Lấy code mới nhất từ GitHub (cả 2 lần đều cần)
import os, subprocess
os.chdir("/kaggle/working")
subprocess.run(["rm", "-rf", "Sign-Language-REL_code"])
subprocess.run(["git", "clone", "https://github.com/linhxm/Sign-Language-REL_code.git"], check=True)
os.chdir("/kaggle/working/Sign-Language-REL_code")
print("cwd:", os.getcwd())

In [ ]:
# ---------- LẦN 1: EXTRACT (Accelerator = None / CPU) ----------
if MODE == "extract":
    import os, sys, subprocess, shutil
    # Thư mục dataset ẢNH: nơi chứa TRỰC TIẾP dev/ train/ test/
    IMG_DIR = "/kaggle/input/datasets/thesisacc2/rwth-phoenix-2014-t"

    # 1) Danh sách tên cần: 5% train (đúng seed loader) + full dev + full test
    subprocess.run([sys.executable, "scripts/make_subset_names.py",
                    "--subset", "0.05", "--out", "/kaggle/working/subset_names.txt"], check=True)

    # 2) Extract CHỈ các sequence đó (không phải cả 8257)
    subprocess.run([sys.executable, "data/extract_poses.py",
                    "--input_dir", IMG_DIR,
                    "--out_dir", "/kaggle/working/poses",
                    "--names_file", "/kaggle/working/subset_names.txt",
                    "--workers", "0"], check=True)

    # 3) Đóng gói thành 1 file .zip để TẢI VỀ từ tab Output
    n = len([f for f in os.listdir("/kaggle/working/poses") if f.endswith(".npz")])
    shutil.make_archive("/kaggle/working/poses_5pct", "zip", "/kaggle/working/poses")
    print(f"\n✅ Extract xong: {n} file .npz")
    print("→ /kaggle/working/poses_5pct.zip : vào tab OUTPUT tải về, tạo New Dataset (upload zip),")
    print("  rồi chạy lại notebook với MODE='train' + Accelerator GPU + Add dataset pose đó.")
else:
    print("bỏ qua extract (MODE != 'extract')")

In [ ]:
# ---------- LẦN 2: TRAIN (Accelerator = GPU T4, Add Input = dataset POSE) ----------
if MODE == "train":
    import os, sys, glob, subprocess
    # Tự tìm thư mục pose đã mount (dataset bạn up từ poses_5pct.zip)
    npz = glob.glob("/kaggle/input/**/*.npz", recursive=True)
    assert npz, "Chưa thấy .npz nào trong /kaggle/input — đã Add dataset pose (từ poses_5pct.zip) chưa?"
    POSE_DIR = os.path.dirname(npz[0])
    print(f"pose_cache_dir = {POSE_DIR}  ({len(npz)} file .npz)")

    # Trỏ config sang thư mục pose vừa mount
    s = open("configs/config.py", encoding="utf-8").read()
    s = s.replace('pose_cache_dir: str = "/kaggle/input/phoenix-poses"',
                  f'pose_cache_dir: str = "{POSE_DIR}"')
    open("configs/config.py", "w", encoding="utf-8").write(s)

    # "core" = chỉ 1 XE + 1 RL (test nhanh luồng, <1h). "all" = FULL ma trận (có thể >12h).
    GROUPS = "core"   # <-- chạy "core" trước cho chắc; xong đổi thành "all"
    subprocess.run([sys.executable, "run_all.py", "--subset", "0.05", "--groups", GROUPS], check=True)
else:
    print("bỏ qua train (MODE != 'train')")

In [ ]:
# Xem kết quả (chỉ ở lần train)
if MODE == "train":
    import os
    p = "/kaggle/working/comparison_table.md"
    if os.path.exists(p):
        print(open(p, encoding="utf-8").read())
    td = "/kaggle/working/report/tables"
    if os.path.isdir(td):
        print("\nBảng báo cáo:", os.listdir(td))